# NPC AI Framework — Colab Training

Runs `training/train_adapter.py` on a Colab GPU instead of the local MX450, which hits a sustained power/thermal throttle (~5W cap) on runs longer than ~1.5-2hrs — see `Docs/TODO.md` Phase 3 for the full incident log (3 configs already tested locally: r=8, r=16, r=8/alpha=32, all showed the same null result on archaic dialect markers; an 8-epoch run to test whether more epochs fixes it couldn't finish locally).

**Before running:** Runtime > Change runtime type > GPU (T4 is fine, free tier).

**What this notebook does:** clone the repo, install deps, log into W&B, run one or more training configs, save adapters to Google Drive (Colab's local disk is wiped when the runtime recycles).

In [ ]:
!nvidia-smi

## 1. Clone the repo

In [ ]:
!git clone https://github.com/spiicez21/Final-Year-Project.git repo
%cd repo

## 2. Install dependencies

Colab ships an old `transformers`/`peft`/`trl` — pin to versions matching what was validated locally so behavior (chat template, SFTConfig API, etc.) matches.

In [ ]:
!pip install -q -U transformers==5.12.1 peft==0.19.1 trl==1.7.0 accelerate==1.14.0 datasets==5.0.0 bitsandbytes==0.49.2 wandb==0.28.0 pyyaml

## 3. Mount Drive (adapters get saved here so they survive runtime resets)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ADAPTERS_DIR = '/content/drive/MyDrive/npc-ai-framework/adapters'
os.makedirs(DRIVE_ADAPTERS_DIR, exist_ok=True)

## 4. W&B login

Paste your API key from wandb.ai/authorize when prompted.

In [ ]:
import wandb
wandb.login()

## 5. Run training

Same `train_adapter.py` used locally, no code changes needed — it already picks CUDA automatically and the bf16 path works on Colab's T4 (also Turing architecture, same as the local MX450, so bf16 is software-emulated there too, but Colab's sustained cooling means no throttling wall).

**Priority queue** (per `Docs/TODO.md` Phase 3 — pick up where local training got stuck):
1. `--epochs 8` at r=8 — the run that couldn't finish locally (stopped at step 120/504 after thermal throttle)
2. `--lora-r 32` — completes the planned rank ablation (r=4/8/16/32)
3. Higher epoch counts (10-15) if 8 still nulls out on archaic markers

Each run takes ~15-25min on a free T4 (vs ~90-180min+ locally) based on the step-time ratio observed before throttling kicked in locally.

In [ ]:
# Run 1: 8 epochs at r=8 — the one that couldn't finish locally
!python training/train_adapter.py --domain medieval --lora-r 8 --epochs 8

In [ ]:
# Copy the finished adapter to Drive before the runtime can recycle it
!cp -r training/adapters/medieval_r8_e8 "{DRIVE_ADAPTERS_DIR}/"

In [ ]:
# Sanity check — same script used locally, same DIALECT_PATTERNS scorer
!python training/quick_inference.py --adapter training/adapters/medieval_r8_e8

In [ ]:
# Run 2: r=32 — completes the planned rank ablation (r=4/8/16/32)
!python training/train_adapter.py --domain medieval --lora-r 32
!cp -r training/adapters/medieval_r32 "{DRIVE_ADAPTERS_DIR}/"
!python training/quick_inference.py --adapter training/adapters/medieval_r32

## 6. If both still null on archaic markers

Escalate epochs further (Colab makes this cheap) before concluding TinyLlama-1.1B can't do this reliably:

In [ ]:
!python training/train_adapter.py --domain medieval --lora-r 8 --epochs 15
!cp -r training/adapters/medieval_r8_e15 "{DRIVE_ADAPTERS_DIR}/"
!python training/quick_inference.py --adapter training/adapters/medieval_r8_e15